In [1]:
import joblib
import os

from utils.pipeline_utils import (
    classify_email, 
    display_result, 
    classify_batch, 
    test_emails,
    classify_email_with_reasoning,
    display_result_with_reasoning,
    explain_spam_prediction,
    display_spam_explanation
)

## Load Pre-trained Models

In [2]:
# Define models directory
models_dir = '../models'

# Load Stage 1: Spam Detection Model
spam_model = joblib.load(os.path.join(models_dir, 'nb_model.joblib'))
spam_vectorizer = joblib.load(os.path.join(models_dir, 'nb_vectorizer.joblib'))

# Load Stage 2: Phishing Type Classification Model
phishing_model = joblib.load(os.path.join(models_dir, 'phishing_nb_model.joblib'))
phishing_vectorizer = joblib.load(os.path.join(models_dir, 'phishing_nb_vectorizer.joblib'))

print(f"Spam model classes: {spam_model.classes_}")
print(f"Phishing model classes: {phishing_model.classes_}")

Spam model classes: [0 1]
Phishing model classes: ['authority_scam' 'credential_harvesting' 'financial_scam'
 'generic_phishing' 'legitimate' 'romance_dating' 'social_engineering'
 'social_engineering_advanced' 'tech_support' 'threats' 'urgency']


## Test the Pipeline with Examples

In [3]:
print("EMAIL CLASSIFICATION RESULTS")

for i, email in enumerate(test_emails, 1):
    print(f"\n>>> Email #{i}")
    result = classify_email(email, spam_model, spam_vectorizer, phishing_model, phishing_vectorizer)
    display_result(result)

EMAIL CLASSIFICATION RESULTS

>>> Email #1
📧 EMAIL CLASSIFICATION RESULT

📝 Email Preview:
    Hi John,
    
    Just wanted to follow up on our meeting yesterday. I've attached the quarterly report 
    as discussed. Let me know if you have any questions.
    
    Best regards,
    Sarah

🔍 Stage 1 - Spam Detection:
    Result:     Ham
    Confidence: 98.8%

📊 FINAL CLASSIFICATION:
    Result:     ✅ Legitimate Email (Ham)
    Risk Level: Low

💡 ADVICE:
    This email appears to be legitimate, but always verify sender addresses.


>>> Email #2
📧 EMAIL CLASSIFICATION RESULT

📝 Email Preview:
    URGENT: Your account has been compromised!
    
    We detected suspicious activity on your account. Click the link below to verify 
    your password and secure your account immediately:
    
    [Ve...

🔍 Stage 1 - Spam Detection:
    Result:     Spam
    Confidence: 89.2%

🎯 Stage 2 - Phishing Type Classification:
    Prediction: credential_harvesting
    Confidence: 57.6%
    2nd Best:   soc

## Batch Classification

In [4]:
batch_results = classify_batch(test_emails, spam_model, spam_vectorizer, phishing_model, phishing_vectorizer)
print("Batch Classification Results:")
batch_results

Batch Classification Results:


,email_preview,spam_detection,spam_confidence,phishing_type,phishing_confidence,alternative_type,alternative_confidence,has_conflict,final_classification,risk_level
0,"Hi John,\n \n Just wanted to follow up o...",Ham,98.8%,N/A,N/A,None,N/A,No,✅ Legitimate Email (Ham),Low
1,URGENT: Your account has been compromised!\n ...,Spam,89.2%,credential_harvesting,57.6%,social_engineering,30.8%,No,🔐 Credential Harvesting,High
2,Congratulations! You have been selected as the...,Spam,99.9%,financial_scam,99.7%,authority_scam,0.1%,No,💰 Financial Scam,High
3,FINAL NOTICE: IRS Tax Violation\n \n Thi...,Spam,50.2%,threats,81.8%,authority_scam,8.4%,No,⚠️ Threat/Extortion,High
4,ALERT: Your computer has been infected!\n \...,Spam,78.7%,social_engineering,56.3%,urgency,9.5%,No,🎭 Social Engineering,Medium


## Interactive Classification (Enter Your Own Email)

In [5]:
# Try your own email
your_email = """
Storage
Don't risk losing your photos, videos, contacts, files and personal private data.

96%
●	Photos	Full
●	Files	Full
●	Family	Full
●	E-mails	Full
●	Device backup	Almost Full
johndoe24 You may not be able to send or receive emails. To continue using cloud services, please free up space or upgrade your storage. Your files are safe for now, but may be deleted if the cloud is not refreshed.

Upgrade now and get an extra 50 GB bonus storage. Don't wait!
This special offer expires in 4 minutes et 39 seconds

UPDATE
"""

result = classify_email(your_email, spam_model, spam_vectorizer, phishing_model, phishing_vectorizer)
display_result(result)

📧 EMAIL CLASSIFICATION RESULT

📝 Email Preview:
    
Storage
Don't risk losing your photos, videos, contacts, files and personal private data.

96%
●	Photos	Full
●	Files	Full
●	Family	Full
●	E-mails	Full
●	Device backup	Almost Full
johndoe24 You may no...

🔍 Stage 1 - Spam Detection:
    Result:     Spam
    Confidence: 96.5%

🎯 Stage 2 - Phishing Type Classification:
    Prediction: urgency
    Confidence: 64.6%
    2nd Best:   credential_harvesting (7.8%)

📊 FINAL CLASSIFICATION:
    Result:     ⏰ Urgency Scam
    Risk Level: High

💡 ADVICE:
    Legitimate organizations rarely demand immediate action. Take your time and verify.



## Classification with Reasoning

The following cells demonstrate the model's reasoning by showing which words/features most influenced the spam/ham classification decision.

In [6]:
# Example 1: Legitimate email with reasoning
legitimate_email = """Hi John,
    
Just wanted to follow up on our meeting yesterday. I've attached the quarterly report 
as discussed. Let me know if you have any questions.

Best regards,
Sarah"""

result = classify_email_with_reasoning(
    legitimate_email, 
    spam_model, spam_vectorizer, 
    phishing_model, phishing_vectorizer,
    show_reasoning=True,
    n_features=10
)
display_result_with_reasoning(result)

📧 EMAIL CLASSIFICATION RESULT

📝 Email Preview:
    Hi John,

Just wanted to follow up on our meeting yesterday. I've attached the quarterly report 
as discussed. Let me know if you have any questions.

Best regards,
Sarah

🔍 Stage 1 - Spam Detection:
    Result:     Ham
    Confidence: 98.8%

📊 FINAL CLASSIFICATION:
    Result:     ✅ Legitimate Email (Ham)
    Risk Level: Low

💡 ADVICE:
    This email appears to be legitimate, but always verify sender addresses.

📊 PREDICTION: HAM
   Confidence: 98.8%
   Spam Probability: 1.2%
   Ham Probability: 98.8%

📈 OVERALL SCORING:
   Total Spam Indicators Score: 0.7269
   Total Ham Indicators Score: 5.1851
   Features Found in Email: 22

🎯 TOP 10 MOST INFLUENTIAL WORDS:
--------------------------------------------------------------------------------

✅ Words pushing towards HAM (Legitimate):
    1. 'discussed           ' ███████████ -0.5649
    2. 'meeting             ' ██████████ -0.5480
    3. 'know questions      ' ██████████ -0.5335
    4.

In [7]:
# Example 2: Spam email with reasoning
spam_email = """CONGRATULATIONS! You have WON $1,000,000 in our international lottery!

Click here NOW to claim your prize! This offer expires in 24 hours!

Send your bank account details and processing fee of $500 to claim.

ACT NOW! LIMITED TIME OFFER! FREE MONEY!!!"""

result = classify_email_with_reasoning(
    spam_email, 
    spam_model, spam_vectorizer, 
    phishing_model, phishing_vectorizer,
    show_reasoning=True,
    n_features=15
)
display_result_with_reasoning(result)

📧 EMAIL CLASSIFICATION RESULT

📝 Email Preview:
    CONGRATULATIONS! You have WON $1,000,000 in our international lottery!

Click here NOW to claim your prize! This offer expires in 24 hours!

Send your bank account details and processing fee of $500 t...

🔍 Stage 1 - Spam Detection:
    Result:     Spam
    Confidence: 99.9%

🎯 Stage 2 - Phishing Type Classification:
    Prediction: financial_scam
    Confidence: 92.4%
    2nd Best:   urgency (7.0%)

📊 FINAL CLASSIFICATION:
    Result:     💰 Financial Scam
    Risk Level: High

💡 ADVICE:
    Never send money or share bank details based on an email request.

📊 PREDICTION: SPAM
   Confidence: 99.9%
   Spam Probability: 99.9%
   Ham Probability: 0.1%

📈 OVERALL SCORING:
   Total Spam Indicators Score: 6.8835
   Total Ham Indicators Score: 0.0514
   Features Found in Email: 28

🎯 TOP 15 MOST INFLUENTIAL WORDS:
--------------------------------------------------------------------------------

📧 Words pushing towards SPAM:
    1. 'claim     

### Standalone Reasoning (Without Full Pipeline)

If you just want to see the spam detection reasoning without running the full phishing classification pipeline:

In [8]:
test_email = "URGENT! Your account will be suspended unless you verify your password NOW!"

explanation = explain_spam_prediction(test_email, spam_model, spam_vectorizer, n_features=15)
display_spam_explanation(explanation, show_text=True, email_text=test_email)


📧 Email Preview:
   URGENT! Your account will be suspended unless you verify your password NOW!

📊 PREDICTION: SPAM
   Confidence: 66.1%
   Spam Probability: 66.1%
   Ham Probability: 33.9%

📈 OVERALL SCORING:
   Total Spam Indicators Score: 1.4141
   Total Ham Indicators Score: 0.7516
   Features Found in Email: 5

🎯 TOP 5 MOST INFLUENTIAL WORDS:
--------------------------------------------------------------------------------

📧 Words pushing towards SPAM:
    1. 'account             ' ████████████ +0.6377
    2. 'urgent              ' █████████ +0.4789
    3. 'verify              ' █████ +0.2975

✅ Words pushing towards HAM (Legitimate):
    1. 'password            ' █████████████ -0.6573
    2. 'unless              ' █ -0.0944

💡 Moderate spam indicators. Some words suggest this might be spam.

